# Free GPU corrected RLVR pilot v2

This notebook runs the existing locked six-condition Qwen2.5-0.5B/GSM8K engineering replication. It checks the corrected sampler/learner policy before each optimizer step, uses the locked 32-task evaluation split, and retains failed attempts. It does not claim a model capability gain or confirmatory result.

Use only a free GPU allocation. If CUDA is unavailable, stop. Do not enable paid compute, switch models/precision/tasks, change thresholds, or reuse this run as independent confirmation.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, shutil, subprocess, sys, time
SOURCE_SHA = "22f24ee"
ATTEMPT = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
CHECKOUT = Path("/content") / ("rvl-corrected-pilot-v2-" + ATTEMPT)
subprocess.run(["git", "clone", "https://github.com/mitukx/Recursive-Verification-Lag.git", str(CHECKOUT)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_SHA], cwd=CHECKOUT, check=True)
actual_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, text=True).strip()
assert actual_sha.startswith(SOURCE_SHA), actual_sha
LOCK = CHECKOUT / "configs/colab_rlvr_pilot_v2.json"
lock = json.loads(LOCK.read_text())
print(json.dumps({"source_sha":actual_sha,"attempt":ATTEMPT,"protocol_sha256":hashlib.sha256(LOCK.read_bytes()).hexdigest(),"lock":lock}, indent=2))

## Verify the free runtime

The runner requires one CUDA GPU and will fail closed without it. A failed allocation is recorded as a feasibility stop; it does not trigger a paid upgrade.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No free CUDA GPU is available; stop without starting paid compute.")
GPU = {"name":torch.cuda.get_device_name(0),"total_memory_bytes":torch.cuda.get_device_properties(0).total_memory,"torch":torch.__version__,"cuda":torch.version.cuda}
print(subprocess.check_output(["nvidia-smi"], text=True))
print(json.dumps(GPU, indent=2))

## Install and verify the locked runtime

The base Colab image may have unrelated package conflicts. They are retained in setup diagnostics. The experiment proceeds only when its pinned imports and CUDA smoke check succeed.

In [ ]:
deps = [f"{name}=={version}" for name, version in lock["dependencies"].items()]
subprocess.run([sys.executable, "-m", "pip", "install", *deps], check=True)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
smoke_code = r'''import importlib.metadata as m, json, sys, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from accelerate import Accelerator
expected=json.loads(sys.argv[1]); actual={name:m.version(name) for name in expected}
assert actual==expected, actual
assert torch.cuda.is_available()
x=torch.ones((16,16),device="cuda"); assert float((x@x).sum())==4096.0
print(json.dumps({"dependencies":actual,"torch":torch.__version__,"gpu":torch.cuda.get_device_name(0),"cuda_smoke":"passed"}))'''
smoke = subprocess.run([sys.executable, "-c", smoke_code, json.dumps(lock["dependencies"])], capture_output=True, text=True)
SETUP = {"gpu":GPU,"pip_check_returncode":pip_check.returncode,"pip_check_stdout":pip_check.stdout,"pip_check_stderr":pip_check.stderr,"import_cuda_smoke_returncode":smoke.returncode,"import_cuda_smoke_stdout":smoke.stdout,"import_cuda_smoke_stderr":smoke.stderr,"paid_compute_started":False}
print(json.dumps(SETUP, indent=2))
if smoke.returncode != 0: raise RuntimeError("Pinned experiment runtime check failed; do not run the study.")
EVIDENCE = Path("/content") / ("rvl-corrected-pilot-v2-evidence-" + ATTEMPT)
EVIDENCE.mkdir(exist_ok=False)
(EVIDENCE/"setup-checks.json").write_text(json.dumps(SETUP, indent=2))
(EVIDENCE/"pip-freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
(EVIDENCE/"notebook-source.json").write_text(json.dumps({"source_sha":actual_sha,"attempt":ATTEMPT,"paid_compute_started":False},indent=2))

## Run the frozen experiment

The runner executes the six locked seed/arm conditions and writes each update and evaluation output immediately. A nonzero exit is retained as a failed attempt.

In [ ]:
RESULT = CHECKOUT / "results" / "colab_rlvr_pilot_v2" / ATTEMPT
RESULT.parent.mkdir(parents=True, exist_ok=False)
run = subprocess.Popen([sys.executable,"-u","scripts/run_colab_rlvr_pilot_v2.py","--output",str(RESULT)],cwd=CHECKOUT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with (EVIDENCE/"runner.log").open("w") as log:
    for line in run.stdout:
        print(line,end=""); log.write(line); log.flush()
returncode=run.wait()
(EVIDENCE/"exit.json").write_text(json.dumps({"returncode":returncode},indent=2))
if RESULT.exists(): shutil.copytree(RESULT,EVIDENCE/"pilot-results")
print("runner exit code:",returncode)

## Recompute the scorecard and download all evidence

The offline checker validates the raw bundle; it is not remote GPU attestation or an independent model evaluator. Keep the run even when it is incomplete or negative.

In [ ]:
if (EVIDENCE/"pilot-results"/"manifest.json").exists():
    audit = subprocess.run([sys.executable,"-m","scripts.summarize_colab_rlvr_pilot_v2",str(EVIDENCE/"pilot-results"),"--output",str(EVIDENCE/"scorecard.json")],cwd=CHECKOUT,capture_output=True,text=True)
    (EVIDENCE/"audit.log").write_text(audit.stdout+audit.stderr)
    if (EVIDENCE/"scorecard.json").exists(): print((EVIDENCE/"scorecard.json").read_text())
    else: print("Scorecard could not be generated; raw run is retained.",audit.returncode,audit.stderr)
else:
    (EVIDENCE/"audit.log").write_text("Raw manifest unavailable; preserve this incomplete attempt.\n")
manifest={str(p.relative_to(EVIDENCE)):hashlib.sha256(p.read_bytes()).hexdigest() for p in EVIDENCE.rglob("*") if p.is_file() and p.name!="bundle-manifest.json"}
(EVIDENCE/"bundle-manifest.json").write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive(str(Path("/content")/("rvl-corrected-pilot-v2-"+ATTEMPT)),"zip",EVIDENCE)
print("Evidence archive:",archive)
from google.colab import files
files.download(archive)